In [1]:
tabla='cbcoc10'
dim='dim_condcit'

In [2]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD_2"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [3]:


oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM cbcoc10", con=connection2)

connection2.close()




In [4]:
base2

,condcitacod,condcitanom,condcitadescor
0,1,NORMAL,N
1,2,ADICIONAL,A


In [5]:
base2.columns

Index(['condcitacod', 'condcitanom', 'condcitadescor'], dtype='object')

In [6]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = "SELECT COUNT(*) FROM cbcoc10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla cbcoc10 antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_cbtci10 ()')
base2.to_sql(name='tmp_cbcoc10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cbtci10 FALSO CON LO OBTENIDO DEL ORACLE

query="""

ALTER TABLE tmp_cbcoc10 
ALTER COLUMN condcitacod  TYPE character(1),
ALTER COLUMN condcitanom  TYPE character(20),
ALTER COLUMN condcitadescor  TYPE character(1);

UPDATE cbcoc10
SET 
condcitacod = t.condcitacod,
condcitanom = t.condcitanom,
condcitadescor = t.condcitadescor

FROM tmp_cbcoc10 t 
WHERE cbcoc10.condcitacod = t.condcitacod AND TRIM(cbcoc10.condcitacod) <> '' AND cbcoc10.condcitacod IS NOT NULL;

INSERT INTO cbcoc10
(condcitacod,condcitanom,condcitadescor) 
SELECT 
condcitacod,condcitanom,condcitadescor

FROM tmp_cbcoc10  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM cbcoc10 
    WHERE cbcoc10.condcitacod = t.condcitacod and cbcoc10.condcitacod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cbcoc10;
SELECT COUNT(*) FROM cbcoc10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla cbcoc10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM cbcoc10", con=connection3)


connection3.close()


Cantidad de filas en la tabla cbcoc10 antes de la inserción: 2
Cantidad de filas en la tabla cbcoc10 después de la inserción: 2
La cantidad de filas insertadas fue de: 0


In [7]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN condcitacod  TYPE character(1),
ALTER COLUMN condcitanom  TYPE character(20),
ALTER COLUMN condcitadescor  TYPE character(1);


INSERT INTO {dim} 

(cod_cci,des_cci,dco_cci) 

SELECT 

condcitacod,condcitanom,condcitadescor

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.cod_cci = t.condcitacod AND
    d.des_cci = t.condcitanom AND
    d.dco_cci = t.condcitadescor
);
"""



c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_condcit antes de la inserción: 0
Cantidad de filas en la tabla dim_condcit después de la inserción: 2
La cantidad de filas insertadas fue de: 2
